Why PPO exists

Core Mechanism

We already know policy gradients and actor-critic, so PPO starts from the problem they have.

1. The problem: policy updates can be too large

Suppose the old policy gives:

π_old(action | state) = 0.50

We collect a trajectory and determine that this action had positive advantage.

A normal policy-gradient update increases its probability:

0.50 → 0.60 → 0.70 → ...

The problem is that a single batch of experience can cause the policy to move too far.

A large policy change can make the new policy very different from the policy that generated the data.

PPO constrains this change.

2. PPO compares old and new policies

For every action from the collected trajectory, calculate:

$$ r_t(\theta) = \frac{\pi_\theta(a_t|s_t)} {\pi_{\text{old}}(a_t|s_t)} $$

This is simply the probability ratio.

Example:

old probability = 0.50
new probability = 0.60

ratio = 0.60 / 0.50
      = 1.2

Interpretation:

ratio = 1 → probability unchanged
ratio > 1 → new policy increased probability
ratio < 1 → new policy decreased probability
3. Combine ratio with advantage

We already calculate an advantage \(A_t\).

If:

A > 0

the action was better than expected, so we want its probability to increase.

If:

A < 0

the action was worse than expected, so we want its probability to decrease.

The basic policy objective is:

$$ r_t A_t $$

Example:

ratio = 1.2
advantage = +2

objective = 1.2 × 2
          = 2.4

So far, this is just importance-weighted policy gradient.

4. PPO clipping

PPO introduces a clipping range:

$$ 1-\epsilon \leq r_t \leq 1+\epsilon $$

Usually:

$$ \epsilon = 0.2 $$

So the useful range is:

0.8 ───────── 1.0 ───────── 1.2

Suppose:

old probability = 0.50
new probability = 0.80

ratio = 1.6

If the advantage is positive, ordinary policy gradient would strongly reward this increase.

PPO clips the ratio:

1.6 → 1.2

The policy therefore doesn't receive additional objective benefit from moving further in that direction.

5. Why the min exists

The actual PPO surrogate objective is:

$$ L^{CLIP} = \mathbb{E} \left[ \min \left( r_tA_t, \operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t \right) \right] $$

The min makes PPO conservative.

Positive advantage

We want:

probability ↑

but not excessively.

Negative advantage

We want:

probability ↓

but not excessively.

So clipping limits how much improvement the optimizer can claim from moving the policy too far.

6. Why PPO keeps the old policy

This is important.

We collect data using:

π_old

Then freeze those probabilities.

For example, suppose our rollout contained:

state = S
action = RIGHT
π_old(RIGHT|S) = 0.40

During optimization, the current policy may change:

π_new(RIGHT|S) = 0.44
π_new(RIGHT|S) = 0.50
π_new(RIGHT|S) = 0.55

We always compare against the original:

0.40

Therefore:

0.44 / 0.40 = 1.10
0.50 / 0.40 = 1.25
0.55 / 0.40 = 1.375

The ratio tells PPO how far the current policy has moved from the policy that generated this data.

That's why we store the old log-probabilities during rollout.

7. PPO training loop

The complete mechanism is:

                OLD POLICY
                    │
                    ▼
             collect rollout
                    │
          ┌─────────┴─────────┐
          ▼                   ▼
       rewards            old log_probs
          │
          ▼
       GAE / advantages
          │
          ▼
    ┌─────────────────┐
    │ PPO optimization│
    └─────────────────┘
          │
          ▼
 current log_probs
          │
          ▼
 ratio = exp(new_log_prob - old_log_prob)
          │
          ▼
      clipping
          │
          ▼
      policy loss
          │
          ▼
      update actor

The critic is trained alongside it using the value targets.

8. One subtle but important point

PPO does not prevent the parameters from changing by more than 20%.

The 0.2 clipping applies to the probability ratio in the PPO objective, not directly to neural-network weights.

That distinction matters.

Yes — the critic \(V(s)\) is independent of the specific action.

It evaluates the state itself:

“How good is it to be in this state?”

So:

V(S1) = 7

doesn't care whether you choose ↑, →, ↓, or ←.

The actor evaluates/chooses actions:

π(↑|S1)
π(→|S1)
π(↓|S1)
π(←|S1)

Then we use the critic's state value as the reference to judge the chosen action.

Critic: state → expected future reward
Actor: state → action probabilities